# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My contract statement.** One row = **one content item (one page)** belonging to one
pseudonymized client, in **one export snapshot**. There are no calendar dates in this
slice: every metric is aggregated over a **trailing 90-day window ending at export
time**, and inside that window two 30-day sub-windows (most recent 30 days vs the 30
days before) are compared to derive the trend. So the "dates" of this contract are
relative — *days -90 to 0* for the totals, *days -30 to 0* vs *days -60 to -31* for
the trend.

The population is the reference prep population: pages with `impressions_90d > 0` and
`content_age_days >= 90`, deduplicated by `content_id`. In this slice all 30,000 rows
already meet those filters (verified below), so the contract covers the full export.

**Output of any analysis under this contract:** a ranked refresh-review queue — one
score per content item — used as decision-support for a content team. Nothing in this
data supports per-day or per-user claims.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find the repo root by walking up from wherever this notebook was opened.
ROOT = Path.cwd()
while not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

RAW = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW)
print("Loaded:", RAW.relative_to(ROOT), "| shape:", df.shape)

# --- Grain: is content_id really the row key? ---
print("rows:", len(df), "| unique content_id:", df["content_id"].nunique(),
      "| duplicated content_id:", int(df["content_id"].duplicated().sum()))

# --- Clients: how many, and how unbalanced? ---
per_client = df.groupby("client_id").size()
print("clients:", df["client_id"].nunique())
print(per_client.describe().round(1))

# --- The label source, counted ---
print()
print(df["trend_direction"].value_counts())
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("declining rows:", int(df["is_declining_label"].sum()),
      "| label rate:", round(df["is_declining_label"].mean(), 4))

# --- Population filters from the reference prep: do they remove anything here? ---
kept = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print()
print("rows surviving prep filters (impressions>0, age>=90):", len(kept), "of", len(df))
print("content_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max())
print("days_with_impressions range:", df["days_with_impressions"].min(), "-", df["days_with_impressions"].max())
print("content_type mix:")
print(df["content_type"].value_counts())


Loaded: data/raw/content_refresh_anonymized.csv | shape: (30000, 44)
rows: 30000 | unique content_id: 30000 | duplicated content_id: 0
clients: 32
count      32.0
mean      937.5
std      1376.4
min         3.0
25%       110.2
50%       567.0
75%      1058.8
max      7008.0
dtype: float64

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
declining rows: 16262 | label rate: 0.5421

rows surviving prep filters (impressions>0, age>=90): 30000 of 30000
content_age_days range: 90 - 564
days_with_impressions range: 1 - 88
content_type mix:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

All 44 columns of the export, sorted. The feature bucket is exactly the reference
model set (`MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in
`scripts/ml_utils.py`) — 18 numeric + 8 categorical = **26 features**, four of the
numeric ones engineered at prep time (`log_*`).

| Bucket | Fields |
|---|---|
| **Feature** (26) | `search_volume`, `competition`, `cpc`, `competition_level`, `content_type`, `main_intent`, `word_count`, `char_count`, `word_count_tier`, `content_age_days`, `age_tier`, `days_since_last_update`, `freshness_tier`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`, and the engineered `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` (log1p of the raw 90-day totals) |
| **Label** (2) | `trend_direction` — the label source: `is_declining_label = (trend_direction == "down")`. `trend_pct` — the label's continuous sibling (the same last-30 vs prev-30 comparison, as a percentage). Neither may ever be a feature. |
| **Context** (2) | `content_id`, `client_id` — pseudonymous IDs. Grouping, joining and client-holdout splitting only; never features. |
| **Excluded** (14) | The six 30-day window counts (`impressions/clicks/sessions_last_30d`, `impressions/clicks/sessions_prev_30d`) — they are the label's ingredients; the trend is computed from them. `provider_used`, `model_used` — which LLM wrote the page is process metadata, not a page property (and 71.5% / 19.1% missing). `age_tier_order` — numeric twin of `age_tier`, redundant. `char_count_tier` — unused twin of the length tiers. `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d` — raw totals superseded by their rates (`engagement_rate`, `scroll_rate`); keeping both would double-count one signal. |

Two scale rules that belong in the contract, because misreading them breaks every
downstream number: **rate columns are ×100 percentages** (`ctr = 0.76` means 0.76%),
and **`avg_position = 0` means "no position data"**, not position zero.


In [2]:
# --- The label derivation is exact: trend_direction alone decides the label ---
print(pd.crosstab(df["trend_direction"], df["is_declining_label"]))

# --- Bucket audit: my four buckets must cover all 44 columns exactly once ---
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

LOG_SOURCES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]
feature_raw = [c for c in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
               if not c.startswith("log_")] + LOG_SOURCES
engineered = [c for c in MODEL_NUMERIC_FEATURES if c.startswith("log_")]
label = ["trend_direction", "trend_pct"]
context = ["content_id", "client_id"]
excluded = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "provider_used", "model_used", "age_tier_order", "char_count_tier",
            "pageviews_90d", "users_90d", "engaged_sessions_90d", "scroll_events_90d"]
covered = feature_raw + label + context + excluded
print()
print("raw feature columns:", len(feature_raw), "| engineered:", engineered)
raw_columns = [c for c in df.columns if c != "is_declining_label"]  # cell 2 added the label
print("buckets cover", len(covered), "raw columns; export has", len(raw_columns),
      "| exact match:", sorted(covered) == sorted(raw_columns))
assert sorted(covered) == sorted(raw_columns)
assert "trend_direction" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
assert "trend_pct" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("label columns confirmed absent from the reference feature lists")


is_declining_label     0      1
trend_direction                
down                   0  16262
flat                1152      0
new                 2236      0
stable              5962      0
up                  4388      0

raw feature columns: 26 | engineered: ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']
buckets cover 44 raw columns; export has 44 | exact match: True
label columns confirmed absent from the reference feature lists


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below back the claims in sections 1–2. What they show, in order:

- **Missingness is patterned, not random.** Keyword-context columns
  (`search_volume`, `competition`, `cpc`, `competition_level`, `main_intent`) are
  missing for **100% of `feedly article` rows** — feedly pages simply have no keyword
  data. A blind `fillna(0)` would silently encode content type into the features.
  `word_count`/`char_count` are missing for 25.7% of rows, concentrated in
  `keyword article` rows (28.3%); comparison and feedly rows are fully measured.
- **The trend window has a hole exactly where the dictionary says.**
  `trend_pct` is blank for 3,388 rows — precisely the rows with
  `impressions_prev_30d = 0` (the `new` + `flat` pages: no previous window exists,
  so no percentage can be computed). The prep step fills these with 0.
- **`avg_position = 0` is a measurement gap**: 1,205 rows (4.0%).
- **`scroll_rate` is blank exactly when `pageviews_90d = 0`** (0.4% of rows) — the
  rate is undefined there, not zero.
- **Ratio columns can exceed 100%** without being errors: `scroll_rate` does on 119
  rows and `ai_traffic_pct` on 23 (numerator and denominator come from different
  measurement systems).
- **Averaging per-row rates lies.** The mean of per-page `ctr` is 0.51%, but the
  true volume-weighted CTR (total clicks / total impressions) is 0.31%. Rates must
  be weighted by their denominators.


In [3]:
# --- Missingness: overall ---
miss = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print("Columns with any missing values (%):")
print(miss[miss > 0])

# --- Missingness is patterned by content_type ---
print()
print("Missing % by content_type:")
print((df.groupby("content_type")[["search_volume", "competition_level", "main_intent",
                                   "word_count"]].agg(lambda s: s.isna().mean()) * 100).round(1))

# --- Window checks ---
print()
print("trend_pct blank rows:", int(df["trend_pct"].isna().sum()),
      "| impressions_prev_30d == 0 rows:", int((df["impressions_prev_30d"] == 0).sum()),
      "| same rows?", df["trend_pct"].isna().equals(df["impressions_prev_30d"] == 0))
print("avg_position == 0 (no position data):", int((df["avg_position"] == 0).sum()), "rows")
print("scroll_rate blank rows:", int(df["scroll_rate"].isna().sum()),
      "| all with pageviews_90d == 0?", bool((df.loc[df["scroll_rate"].isna(), "pageviews_90d"] == 0).all()))

# --- Rate sanity: ratios above 100 exist and are legitimate ---
print()
print("scroll_rate > 100 rows:", int((df["scroll_rate"] > 100).sum()),
      "| ai_traffic_pct > 100 rows:", int((df["ai_traffic_pct"] > 100).sum()),
      "| ctr max:", df["ctr"].max())

# --- Rates need denominators: mean of per-row CTR vs the true weighted CTR ---
print()
print("mean of per-row ctr (%):", round(df["ctr"].mean(), 2),
      "| weighted CTR = total clicks / total impressions (%):",
      round(100 * df["clicks_90d"].sum() / df["impressions_90d"].sum(), 2))


Columns with any missing values (%):
provider_used        71.5
word_count           25.7
char_count           25.7
char_count_tier      25.7
word_count_tier      25.7
model_used           19.1
trend_pct            11.3
competition_level     8.7
cpc                   8.2
search_volume         8.2
competition           8.2
main_intent           7.9
scroll_rate           0.4
dtype: float64

Missing % by content_type:
                    search_volume  competition_level  main_intent  word_count
content_type                                                                 
comparison article            0.0                0.0          0.0         0.0
feedly article              100.0              100.0        100.0         0.0
keyword article               1.4                1.9          1.0        28.3

trend_pct blank rows: 3388 | impressions_prev_30d == 0 rows: 3388 | same rows? True
avg_position == 0 (no position data): 1205 rows
scroll_rate blank rows: 125 | all with pageviews_90d == 0? 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this slice can never tell me, stated plainly:

1. **Nothing calendar-shaped.** There is no date column — all windows are relative to
   export time. I cannot build time-based splits, check seasonality, align pages to
   real-world events, or follow a page after a refresh to see whether it recovered.
   Any "did the refresh work?" question is unanswerable here (and would be a causal
   claim anyway).
2. **Nothing about the future.** Features (90-day aggregates) and the label (last-30
   vs prev-30 days) are measured over the *same* trailing window. A model on this
   contract ranks pages that are **observed declining now** — decision-support for
   prioritising review — it does not forecast a future window. A forward-looking
   version needs the warehouse release with strictly earlier feature windows.
3. **Client-level fairness.** History is deeply unbalanced: 32 clients contribute
   between 3 and 7,008 rows each (median 567; 7 clients have fewer than 50 rows).
   Client-holdout scores swing with *which* clients are held out, and tiny clients
   can neither train nor validate anything on their own.
4. **Why a page declined.** There is no page text, no titles, no keywords, no SERP or
   competitor data in the export (by design — see `DATA_USE.md`). I can measure
   *that* a page is declining and what it looks like, never *why*.
5. **Clean tier readings.** `position_tier = top_3` mixes two populations: 1,116
   rows with a measured position ≤ 3 (median 53 impressions/90d — small denominators)
   and all 1,205 `avg_position = 0` no-data rows (median 1 impression). Any top_3
   statistic that doesn't split these is meaningless.
6. **Pages with no audience.** The population starts at `impressions_90d > 0` and
   `content_age_days >= 90`. Brand-new or invisible pages are outside the contract
   entirely.


In [4]:
# --- Limit 1: no calendar information exists in the slice ---
date_cols = [c for c in df.columns
             if {"date", "month", "year", "timestamp"} & set(c.split("_"))]
datetime_cols = df.select_dtypes(include=["datetime64"]).columns.tolist()
print("columns carrying a date/month/year field:", date_cols,
      "| datetime-typed columns:", datetime_cols)

# --- Limit 3: client imbalance, in numbers ---
sizes = df.groupby("client_id").size()
print("rows per client: min", sizes.min(), "| median", int(sizes.median()),
      "| max", sizes.max(), "| clients with < 50 rows:", int((sizes < 50).sum()))

# --- Limit 5: position_tier 'top_3' is two different populations ---
top3 = df[df["position_tier"] == "top_3"]
measured = top3[top3["avg_position"] > 0]
nodata = top3[top3["avg_position"] == 0]
print()
print("position_tier top_3 rows:", len(top3))
print("  measured position <= 3:", len(measured),
      "| median impressions_90d:", measured["impressions_90d"].median())
print("  avg_position == 0 (no data):", len(nodata),
      "| median impressions_90d:", nodata["impressions_90d"].median())

# --- Limit 2: how much of the label window is even defined? ---
print()
print("rows with a defined trend window (impressions_prev_30d > 0):",
      int((df["impressions_prev_30d"] > 0).sum()), "of", len(df))


columns carrying a date/month/year field: [] | datetime-typed columns: []
rows per client: min 3 | median 567 | max 7008 | clients with < 50 rows: 7

position_tier top_3 rows: 2321
  measured position <= 3: 1116 | median impressions_90d: 53.0
  avg_position == 0 (no data): 1205 | median impressions_90d: 1.0

rows with a defined trend window (impressions_prev_30d > 0): 26612 of 30000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.